# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
from typing import Optional, Literal
from IPython.display import display, Markdown

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment
load_dotenv()

class Model:
    def __init__(self):
        self.client_oai = OpenAI(api_key=os.getenv("OPENAI_KEY"))
        self.ollama_base_url = f"{os.getenv('OLLAMA_URL')}/api/chat"

    def _prompt_llama(self, text: str):
        response = requests.post(
            self.ollama_base_url,
            json={
                "model": MODEL_LLAMA,
                "messages": [{"role": "user", "content": text}],
                "stream": True
            },
            stream=True
        )
        for line in response.iter_lines():
            if not line:
                continue
                
            data = line.decode("utf-8")
            if data.strip() == "":
                continue
                
            if data.startswith("data:"):
                data = data[5:].strip()
                
            yield data

    def _prompt_oai(self, question: str):
        stream = self.client_oai.chat.completions.create(
            model=MODEL_GPT,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an advanced reasoning and explanation engine. "
                        "You write with precision, clarity, and conciseness. "
                        "You can explain Python, algorithms, code design, and systems-level behavior "
                        "with technical rigor, while being straight to the point."
                    ),
                },
                {"role": "user", "content": question},
            ],
            stream=True,
        )
        return stream

    def prompt(self, question: str, model: Optional[Literal[MODEL_GPT, MODEL_LLAMA]] = MODEL_GPT):
        if "gpt" in model:
            stream = self._prompt_oai(question)
            buffer = []
            for event in stream:
                if event.choices and event.choices[0].delta.content:
                    chunk = event.choices[0].delta.content
                    buffer.append(chunk)
                    yield chunk
                    
            output = "".join(buffer)

        else:
            stream = self._prompt_llama(question)
            buffer = []
            for chunk in stream:
                try:
                    import json
                    data = json.loads(chunk)
                    content = data.get("message", {}).get("content", "")
                    
                    if content:
                        buffer.append(content)
                        yield content
                        
                except Exception as e:
                    print("An error occured", e)
                    continue
                    
            output = "".join(buffer)

        display(Markdown(output))

In [4]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get gpt-4o-mini to answer, with streaming
model = Model()

for token in model.prompt(question, model=MODEL_GPT):
    print(token, end="", flush=True)


The provided code snippet utilizes the `yield from` construct in Python, alongside a generator expression that creates a set of unique authors from a list of book dictionaries. Here's a breakdown of what each part does:

1. **Generator Expression**: The expression `{book.get("author") for book in books if book.get("author")}` is a set comprehension that:
   - Iterates over each `book` in the iterable `books`.
   - For each `book`, it calls `book.get("author")` to retrieve the value associated with the "author" key.
   - The `if book.get("author")` condition filters out any `None` or falsy values, ensuring only valid author names are included.

2. **Output**: The expression produces a set of unique authors, as sets inherently do not allow duplicate values.

3. **Yield from**: The `yield from` statement is used to yield all values from the set directly. This means that each unique author from the resulting set is yielded one at a time, allowing for an iterable interface that can be consu

The provided code snippet utilizes the `yield from` construct in Python, alongside a generator expression that creates a set of unique authors from a list of book dictionaries. Here's a breakdown of what each part does:

1. **Generator Expression**: The expression `{book.get("author") for book in books if book.get("author")}` is a set comprehension that:
   - Iterates over each `book` in the iterable `books`.
   - For each `book`, it calls `book.get("author")` to retrieve the value associated with the "author" key.
   - The `if book.get("author")` condition filters out any `None` or falsy values, ensuring only valid author names are included.

2. **Output**: The expression produces a set of unique authors, as sets inherently do not allow duplicate values.

3. **Yield from**: The `yield from` statement is used to yield all values from the set directly. This means that each unique author from the resulting set is yielded one at a time, allowing for an iterable interface that can be consumed by whatever calls the generator.

### Purpose and Use Case

- **Purpose**: The code is effectively creating an iterator for unique authors of books from a list of book entries, while gracefully handling cases where the author might be missing or unspecified.

- **Use Case**: This is useful in scenarios where you need to process a collection of books and want to get a distinct list of authors, such as when generating reports, displaying a list of authors, or preparing data for further analysis.

### Example

Given a list of books like:
```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": None},
    {"title": "Book 4", "author": "Author A"},
]
```

The code would yield:
- "Author A"
- "Author B"

Thus, the output will contain each author exactly once.

In [6]:
# Get Llama 3.2 to answer

for token in model.prompt(question, model=MODEL_LLAMA):
    print(token, end="", flush=True)

MissingSchema: Invalid URL 'None/api/chat': No scheme supplied. Perhaps you meant https://None/api/chat?